# Session 1: What a machine means when it says it learns

**259.075 Data-integrated Algorithmic Design Processes II** · Session 1 of 7

<img src="img/wien-zinshaus.jpg" width="620">

*A late nineteenth century apartment house in Vienna, Lange Gasse 25. Photo: Funke,
[Wikimedia Commons](https://commons.wikimedia.org/wiki/File:1080_Lange_Gasse_25.jpg),
CC BY-SA 4.0. Six hundred and ninety of the 1073 buildings in our table were built in the
same decades, between 1860 and 1918.*

Take a building in Vienna, say a corner house in the seventh district, and ask what it needs
for heating. The usual way to answer takes an afternoon: you enter the geometry, the wall
build-ups, the windows, you run a simulation, you get a number. Do that for a hundred design
variants and you have spent a month.

> A model is a fast answer to a question that used to be slow.

We spend this term building something that answers in a millisecond instead, well enough to
be useful while you are still drawing. Today we build the smallest such thing there is: one
number in, one number out, and a rule in between that the machine finds by itself.

The buildings in our table come from the open data of the City of Vienna, layer
`GEBAEUDETYPOGD`: one polygon per building, with its construction period. The heating demand
does not. No open dataset here gives measured consumption per building, so that column is
computed from the wall U-value of the period and the shape of the footprint. It matters more
than it sounds, and we return to it at the end. The column names are German; the box below
translates them.

<div style="border-left: 5px solid #d8a24a; background-color: #fdf6e3; color: #222; padding: 10px 16px; margin: 6px 0;">

**💡 About the data**

The table loaded below is ours, not the city's. From the city layer come the polygon, the
construction period and the building type. Area, perimeter and compactness are measured from the
polygon. The U-value is a textbook value assumed per construction period, not a survey of the
actual walls, and the heating demand is computed from it and the compactness, in
`data/fetch_ogd.py`. Of the columns, only `id`, `bauperiode` and `bautyp` come from the city;
the rest is our doing.

| column | German | meaning |
|---|---|---|
| `id` | | the city's number for the building |
| `bauperiode` | Bauperiode | construction period, as the city records it; *vor* means before, *nach* after |
| `startjahr` | Startjahr | a year taken from the period's name, used for colouring |
| `bautyp` | Bautyp | the city's code for the type of building |
| `flaeche_m2` | Fläche | footprint area in m² |
| `umfang_m` | Umfang | perimeter of the footprint in m |
| `kompaktheit` | Kompaktheit | compactness, explained in a moment |
| `u_wert` | U-Wert | U-value of the wall in W/m²K: how much heat passes through it, so higher means a leakier wall |
| `hwb` | Heizwärmebedarf | heating demand in kWh per m² and year, the computed column |
| `lat`, `lon` | | position on the map |

</div>

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

df = pd.read_csv("../data/gebaeude_wien.csv")       # Gebäude Wien: Vienna buildings
print(df.shape)
df.head()

## Guessing first

Run the next cell. One point per building: compactness on the horizontal axis, heating demand
on the vertical.

Compactness is the perimeter of the footprint divided by the square root of its area. A round
building sits at 3.5 (perimeter 2πr over √(πr²) is 2√π), a long slab far above. Hover over
three or four points and see which construction period they are from.

> Compactness: perimeter over square root of area. Dimensionless, so a shed and a block can be
> compared.

One question you should be able to answer before reading on. Hover over two points with almost
the same compactness but clearly different heights, and compare their `u_wert`: why is their
heating demand different, and what does the plot not show you?

<details>
<summary>Answer</summary>

Their U-values differ. Two houses of the same shape with different walls have different demand,
so the column depends on something besides the shape; the box above says so. The horizontal
axis shows only the shape, so the points cannot sit on one line. They sit in bands, one per
U-value. The colour gives a hint, because the U-value is set per construction period, but
several periods share the same value and therefore the same band.

</details>

In [ ]:
fig = px.scatter(df, x="kompaktheit", y="hwb", color="startjahr",
                 hover_data=["id", "bauperiode", "bautyp", "flaeche_m2", "u_wert"],
                 labels={"kompaktheit": "compactness (perimeter / sqrt area)",
                         "hwb": "heating demand, computed [kWh/m2a]",
                         "startjahr": "period from"},
                 color_continuous_scale="Viridis", height=520)
fig.update_traces(marker=dict(size=7, opacity=0.75))
fig.show()

The same houses on the map. The table covers four parts of Vienna, not the whole city. Find a
building you know, the one you live in or the institute if it is there, otherwise any you
recognise, hover over it and note its id. You will meet it again at the end of the session.

In [ ]:
fig = px.scatter_map(df, lat="lat", lon="lon", color="hwb", size="flaeche_m2",
                     color_continuous_scale="Turbo", zoom=11, height=520,
                     hover_data=["id", "bauperiode", "kompaktheit", "u_wert"],
                     map_style="carto-positron")
fig.show()

## Your line, by hand

Run the code cell below this text. It draws the buildings, a red line, and under the plot a
slider marked "w =". Drag it. The line `hwb = w · compactness + b` moves with it. `w` is
its slope: how much the predicted demand rises when compactness goes up by one. `b` lifts or
lowers the whole line; it stays fixed at 110 for now, a value picked by eye so that the line
starts somewhere near the cloud. The title shows one number: the mean squared error, the
average of the squared vertical distances between line and points.

Find the smallest value you can and write it down. We need it in twenty minutes.

<details>
<summary>Answer</summary>

At w = 8 the title shows an error of 53. Neighbouring values are worse in both directions. That
is the best of the slider's whole-number values with b held at 110; with b free as well, the
loop below gets lower.

</details>

In [ ]:
def mse(w, b, x, y):
    return float(np.mean((w * x + b - y) ** 2))

x = df["kompaktheit"].to_numpy()
y = df["hwb"].to_numpy()
xs = np.linspace(x.min(), x.max(), 50)
b_fix = 110.0
ws = np.arange(0, 31, 1)

frames = [go.Frame(name=f"{w}",
                   data=[go.Scatter(x=xs, y=w * xs + b_fix, mode="lines",
                                    line=dict(width=4, color="crimson"))],
                   traces=[1],
                   layout=go.Layout(title=f"w = {w},  b = {b_fix:.0f},  error = {mse(w, b_fix, x, y):.0f}"))
          for w in ws]

fig = go.Figure(
    data=[go.Scatter(x=x, y=y, mode="markers", name="buildings",
                     marker=dict(size=6, opacity=0.5, color="#4c78a8")),
          go.Scatter(x=xs, y=ws[0] * xs + b_fix, mode="lines", name="your model",
                     line=dict(width=4, color="crimson"))],
    frames=frames)
fig.update_layout(
    height=560, xaxis_title="compactness", yaxis_title="heating demand, computed",
    title=f"w = {ws[0]},  b = {b_fix:.0f},  error = {mse(ws[0], b_fix, x, y):.0f}",
    sliders=[dict(active=0, currentvalue=dict(prefix="w = "),
                  steps=[dict(method="animate", label=f"{w}",
                              args=[[f"{w}"], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
                         for w in ws])])
fig.show()

## The neuron

<img src="img/linear-regression.png" width="460">

*What you are doing by hand is two hundred years old. Legendre and Gauss published it around
1805 as the method of least squares. Figure: Sewaqu,
[Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Linear_regression.svg), public
domain.*

The line you were moving is

    hwb = w · compactness + b

and in the language of the field this is a *neuron*: inputs, one weight per input (`w`), a
bias term (`b`), an output. Every network you will meet later, including the ones that draw
pictures, is built from this, plus one more ingredient that session 2 adds.

Squaring the distances punishes a few large mistakes more than many small ones. That is a
choice, not a law, and we meet a different one in session 6.

## Downhill in the dark

Go back to the slider under your line plot, move it two steps to the right and watch the
error in the title. If it grows you went the wrong
way, if it falls keep going, and move one tick at a time once it flattens. That rule finds the bottom
without any knowledge of the shape of the landscape, only of whether it rises or falls where you
stand.

> The slope of the error where you stand has a name: the gradient. It points the way the error
> rises fastest, so you step against it.

<img src="img/gradient-descent.gif" width="420">

*The same idea on a two-dimensional landscape: the point keeps stepping downhill until it sits
at the bottom. Animation: Jacopo Bertolotti,
[Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Gradient_descent.gif), CC0.*

The next two cells draw that landscape, first over `w` alone, then as a surface over `w` and
`b`. Turn the surface until you see the valley from the side. It is narrow and it lies at an
angle, and that costs us dearly in a minute.

In [ ]:
w_grid = np.linspace(0, 30, 200)
losses = [mse(w, b_fix, x, y) for w in w_grid]

fig = go.Figure(go.Scatter(x=w_grid, y=losses, mode="lines", line=dict(width=3)))
fig.update_layout(height=420, xaxis_title="w", yaxis_title="mean squared error",
                  title=f"error over w, at fixed b = {b_fix:.0f}")
fig.show()

In [ ]:
W, B = np.meshgrid(np.linspace(0, 30, 60), np.linspace(60, 170, 60))
Z = np.array([[mse(w, b, x, y) for w in W[0]] for b in B[:, 0]])

fig = go.Figure(go.Surface(x=W[0], y=B[:, 0], z=Z, colorscale="Viridis", opacity=0.92))
fig.update_layout(height=620, title="error landscape over w and b",
                  scene=dict(xaxis_title="w", yaxis_title="b", zaxis_title="error"))
fig.show()

## Your part

The gradient for this model is given below. `gradient` returns two numbers, `gw` and `gb`: how
steeply the error rises when `w` or `b` goes up. The loop is yours: two lines, one step against
the gradient for `w`, one for `b`. The code uses German names: `lernrate` is the learning rate,
`schritte` the number of steps, `fehler` the error, `verlauf` the history. A step against the
gradient means: new value = old value − learning rate × slope. With w = 10, slope 4 and learning rate 0.1, the next w is 9.6. In Python that is
`w = w - lernrate * gw`. Replace the line with `raise NotImplementedError` by your two lines.

<details>
<summary>Answer</summary>

    w = w - lernrate * gw
    b = b - lernrate * gb

</details>

`fit` starts with a flat line at the average demand and keeps `(w, b, error)` after every step in `verlauf`; the learning rate 0.015
and the 3000 steps are our choice for now.

The derivation of the gradient is at the end of this notebook, under Homework. Open it if you
want it; it is not examined. If you are stuck for ten minutes, the solution is released, see
the course rules; nobody loses the actual task over a bracket.

The cell after the loop draws the *loss curve*: the error, our mean squared error, after every
step. Its vertical axis is logarithmic: equal distances stand for equal ratios, so going from
10 to 100 is as far as going from 100 to 1000, and a curve that falls steadily on it is
shrinking by the same factor again and again.

In [ ]:
def gradient(w, b, x, y):
    fehler = w * x + b - y
    return float(2 * np.mean(fehler * x)), float(2 * np.mean(fehler))


def fit(x, y, lernrate=0.015, schritte=3000, w0=0.0, b0=None):
    w, b = w0, float(np.mean(y)) if b0 is None else b0
    verlauf = [(w, b, mse(w, b, x, y))]
    for _ in range(schritte):
        gw, gb = gradient(w, b, x, y)

        # YOUR TURN: one step against the gradient, two lines, one for w and one for b.
        # The step length is the learning rate.
        raise NotImplementedError("update rule missing")

        verlauf.append((w, b, mse(w, b, x, y)))
    return w, b, np.array(verlauf)


w_end, b_end, verlauf = fit(x, y)
print(f"w = {w_end:.1f}, b = {b_end:.1f}, error = {verlauf[-1, 2]:.1f}")

In [ ]:
fig = go.Figure(go.Scatter(y=verlauf[:, 2], mode="lines", line=dict(width=3)))
fig.update_layout(height=380, xaxis_title="step", yaxis_title="error", yaxis_type="log",
                  title=f"loss curve, learning rate 0.015, final error {verlauf[-1, 2]:.1f}")
fig.show()

Left: the error landscape seen from above. Each contour line joins pairs of `w` and `b` with
the same error, and the red path is the sequence of pairs the loop visited. Right: the line
settling into the cloud. Press the Play button above the left plot.

Compare the final error with the smallest value you found by hand.

<details>
<summary>Answer</summary>

The loop ends at 50.3, the slider's best was 53 at w = 8. The loop gets lower because it moves
`b` as well.

</details>

In [ ]:
from plotly.subplots import make_subplots

bilder = np.linspace(0, len(verlauf) - 1, 60).astype(int)   # Bilder: frames, the steps shown
fig = make_subplots(rows=1, cols=2, subplot_titles=("path through the error landscape",
                                                    "model in the data"))
fig.add_trace(go.Contour(x=W[0], y=B[:, 0], z=Z, colorscale="Viridis", showscale=False),
              row=1, col=1)
fig.add_trace(go.Scatter(x=[verlauf[0, 0]], y=[verlauf[0, 1]], mode="lines+markers",
                         marker=dict(size=7, color="crimson"), name="parameters"), row=1, col=1)
fig.add_trace(go.Scatter(x=x, y=y, mode="markers", name="buildings",
                         marker=dict(size=5, opacity=0.4, color="#4c78a8")), row=1, col=2)
fig.add_trace(go.Scatter(x=xs, y=verlauf[0, 0] * xs + verlauf[0, 1], mode="lines",
                         line=dict(width=4, color="crimson"), name="model"), row=1, col=2)

fig.frames = [go.Frame(name=str(i),
                       data=[go.Scatter(x=verlauf[:i + 1, 0], y=verlauf[:i + 1, 1],
                                        mode="lines+markers",
                                        marker=dict(size=5, color="crimson")),
                             go.Scatter(x=xs, y=verlauf[i, 0] * xs + verlauf[i, 1],
                                        mode="lines", line=dict(width=4, color="crimson"))],
                       traces=[1, 3])
              for i in bilder]
fig.update_layout(height=560, showlegend=False,
                  updatemenus=[dict(type="buttons", x=0.02, y=1.15,
                                    buttons=[dict(label="Play", method="animate",
                                                  args=[None, dict(frame=dict(duration=60, redraw=True),
                                                                   fromcurrent=True)]),
                                             dict(label="Stop", method="animate",
                                                  args=[[None], dict(mode="immediate")])])])
fig.update_xaxes(title_text="w", row=1, col=1)
fig.update_yaxes(title_text="b", row=1, col=1)
fig.update_xaxes(title_text="compactness", row=1, col=2)
fig.update_yaxes(title_text="heating demand", row=1, col=2)
fig.show()

## Now break it

The next cell runs the same loop with a learning rate of 0.04 instead of 0.015. The homework
at the end asks you to find exactly where between the two it tips over.

> If the loss curve rises steadily with everything else correct, the learning rate is too large.
> This is the most common failure in the whole field.

Three thousand steps for a straight line is embarrassing. If you finish early: compute the exact
solution with `np.polyfit(x, y, 1)`, which solves the least-squares problem directly, compare it
with what the loop found, and think about why the loop took so long.

<details>
<summary>Answer</summary>

`np.polyfit` gives w = 8.506 and b = 108.782, error 50.343; the loop ends at w = 8.495 and
b = 108.846, error 50.343. The valley you turned around earlier is narrow and tilted. A step
large enough to move quickly along it would jump across it, so the steps have to be small, and
small steps take long to travel down its length. Session 2 deals with that.

</details>

In [ ]:
_, _, wild = fit(x, y, lernrate=0.04, schritte=60)

fig = go.Figure(go.Scatter(y=wild[:, 2], mode="lines+markers", line=dict(width=3, color="crimson")))
fig.update_layout(height=380, xaxis_title="step", yaxis_title="error", yaxis_type="log",
                  title="loss curve, learning rate 0.04")
fig.show()

## What the model actually learned

The error stops at about 50, and the test cell below checks that no straight line does better.
That is not the fault of the loop. A building has more to it than the shape of its footprint,
and the U-value of the wall, which also goes into the computed demand, sits in the data but not
in the model. Session 2 hands it over.

Now the map again, coloured by *residual*: computed demand minus predicted demand, one number
per building. Red where the model overestimates, blue where it underestimates. Find the house you picked at the beginning. You will see a pattern by
construction period, and you should, because the target was computed from the construction
period in the first place.

> Our model recovers a calculation rule that someone wrote down. It has not learned how a house
> behaves. Everything in this course stands or falls on that difference.

This is deliberate: a rule has no measurement error, so everything in the first weeks can be
checked exactly. From session 3 we add noise, and from there telling signal from accident is
the whole problem.

In [ ]:
df["vorhersage"] = w_end * x + b_end                 # Vorhersage: prediction
df["residuum"] = df["hwb"] - df["vorhersage"]        # Residuum: residual

fig = px.scatter_map(df, lat="lat", lon="lon", color="residuum", zoom=11, height=560,
                     color_continuous_scale="RdBu", range_color=(-30, 30),
                     hover_data=["id", "bauperiode", "u_wert", "kompaktheit", "vorhersage", "hwb"],
                     map_style="carto-positron")
fig.show()

## Tests

This cell has to run green before you submit. It recomputes what your `fit` returns and
compares it with the best line there is for this data. These are the same tests the grader
uses, there with different buildings.

In [ ]:
w_t, b_t, verlauf_t = fit(x, y)

eigener_fehler = float(np.mean((w_t * x + b_t - y) ** 2))            # your error
bestes_w, bestes_b = np.polyfit(x, y, 1)                             # best w and b
bester_fehler = float(np.mean((bestes_w * x + bestes_b - y) ** 2))   # best possible error

assert np.isfinite([w_t, b_t]).all(), "parameters ran away"
assert abs(eigener_fehler - verlauf_t[-1, 2]) < 1e-6, "reported error does not match the parameters"
assert verlauf_t[-1, 2] < verlauf_t[0, 2], "the error must fall during the run"
assert eigener_fehler < 1.05 * bester_fehler, (
    f"still too far off: {eigener_fehler:.1f} against {bester_fehler:.1f} at best")
print(f"all green, error {eigener_fehler:.2f}, best possible {bester_fehler:.2f}")

## Where you stand now

You can look at a cloud of buildings and say what a model would do with it. You can name the
parts of a neuron, read an error landscape, and recognise a learning rate that is too large by
the shape of the loss curve alone. And you can say, about your own result, what it does not
mean: our model reproduces a calculation rule, it has not learned how a house behaves.

Next session we hand the model the U-value as a second input. The error collapses, and the
reason why that is less impressive than it looks is the first thing we discuss.

---

### Terms to learn

These are the words from this session. They are in the flashcard app, together with the terms
from the sessions that follow, at
[gabriel-wurzer.github.io/data2/webapps/lernkarten](https://gabriel-wurzer.github.io/data2/webapps/lernkarten/).
Add it to your home screen; it works offline and takes four minutes. *Lernkarten* is German for
flashcards.

| term | in one sentence |
|---|---|
| model | a fast answer to a question that used to be slow |
| neuron | inputs, one weight each, a bias, one output |
| weight, bias | the numbers the machine changes while it learns |
| parameter | weight and bias together; what is learned, not what you set |
| loss, mean squared error | one number saying how wrong the model is on the data |
| gradient | the slope of the loss where you stand; it points uphill |
| gradient descent | repeatedly stepping against the gradient |
| learning rate | how far you step; too large and the loss rises |
| least squares | the closed-form solution Legendre and Gauss published in 1805 |
| residual | computed (later: measured) minus predicted, for one building |

## Homework

Ten points: four for the run, six for what you make of it. Submit this notebook, run from top
to bottom, with your text in the last cell. If you used lines from the solution notebook, say
so in one line at the top of your text. That costs nothing, see the course rules; a submission
that is missing costs everything.

Above this heading, the only thing to fill in is the `YOUR TURN` gap from class, by yourself
or from the solution. Below it, change only the lines under `# YOUR CHOICE`, and write your
answers into the last cell. Before you submit, choose Kernel, Restart Kernel and Run All Cells,
and check that everything still runs.

The four points for the run: the notebook runs from top to bottom without an error (one), what
the task asks you to show is there (one), and every number you give about your own run either
appears in your output, rounded as printed, or is worked out or counted from it in a step you
write down (two). Numbers the questions give you, numbers in an example from outside the
notebook, and a forecast you mark as your own expectation with a reason, are exempt. The six points for the interpretation are one per question below, no halves. A
sentence that would fit any model earns nothing; a sentence that names what you saw and says
what it means earns the point, also with borrowed code.

### The derivation

<details>
<summary>Where the two lines in <code>gradient</code> come from. Open it if you want it; it is
not examined.</summary>

The error is the mean of the squared misses:

    E(w, b) = (1/n) · Σ (w · x_i + b − y_i)²

Call the bracket `e_i`, the miss on building i. Each term is a square, so its slope is two
times the miss times the slope of the miss itself. The miss changes by `x_i` when w goes up by
one, and by 1 when b goes up by one:

    ∂E/∂w = (2/n) · Σ e_i · x_i
    ∂E/∂b = (2/n) · Σ e_i

That is `2 * np.mean(fehler * x)` and `2 * np.mean(fehler)`, nothing more.

</details>

### The task

1. **Where it tips over.** The first cell runs the loop for 300 steps at each learning rate in
   the list and says, for each, whether the error is still falling at the end or rising. Start
   with the list as it is. Then replace it by rates 0.001 apart, between the last one that falls
   and the first one that rises, and run again. You are done when you have two rates 0.001
   apart, the lower falling and the higher rising. Keep that figure.
2. **Your loop against the exact answer.** The second cell needs no change. It prints your
   loop's result next to the least-squares line, and the numbers you need for questions 1 and 4.
3. **Your building.** Put the id you clicked on the map at the start of the session into the
   third cell. If you did not note one, open the map again and pick a building you know. The
   cell prints its residual and, for comparison, the buildings of almost the same compactness.

In [ ]:
# YOUR CHOICE: the learning rates (lernraten) you try. Start with these, then close in,
# 0.001 apart. lauf is one run's history, richtung its direction at the end.
lernraten = [0.01, 0.02, 0.03, 0.04]

fig = go.Figure()
with np.errstate(over="ignore", invalid="ignore"):
    for lr in lernraten:
        _, _, lauf = fit(x, y, lernrate=lr, schritte=300)
        richtung = "rising" if lauf[-1, 2] > lauf[-51, 2] else "falling"
        fig.add_trace(go.Scatter(y=lauf[:, 2], mode="lines", name=f"learning rate {lr}"))
        print(f"learning rate {lr:<6}: error after 300 steps {lauf[-1, 2]:.4g}, "
              f"over the last 50 steps {richtung}")
fig.update_layout(height=420, xaxis_title="step", yaxis_title="error", yaxis_type="log",
                  title="where the run tips over")
fig.show()

In [ ]:
bestes_w, bestes_b = np.polyfit(x, y, 1)
fehler_loop = mse(w_end, b_end, x, y)       # the error of your loop
print(f"your loop    : w = {w_end:.3f}, b = {b_end:.3f}, error {fehler_loop:.3f}")
print(f"least squares: w = {bestes_w:.3f}, b = {bestes_b:.3f}, "
      f"error {mse(bestes_w, bestes_b, x, y):.3f}")
print(f"typical miss of your loop, the square root of its error: {np.sqrt(fehler_loop):.1f} kWh/m2a")
print(f"standard deviation of the computed demand: {y.std():.1f} kWh/m2a, the typical miss "
      f"of a model that always guesses the average")
print(f"compactness in the data: {x.min():.2f} to {x.max():.2f}")
print(f"your line at compactness 15: {w_end * 15 + b_end:.1f} kWh/m2a, "
      f"highest demand in the data {y.max():.1f}")

In [ ]:
# YOUR CHOICE: the id of your building (meine_id), from the map at the start of the session.
# haus is its row in the table, nah the buildings of nearly the same compactness.
meine_id = None

assert meine_id is not None, "put the id of your building into meine_id"
haus = df[df["id"] == meine_id].iloc[0]
print(f"building {meine_id}, {haus['bauperiode']}, compactness {haus['kompaktheit']:.2f}, "
      f"u-value {haus['u_wert']:.2f}")
print(f"predicted {haus['vorhersage']:.1f}, computed {haus['hwb']:.1f}, "
      f"residual {haus['residuum']:+.1f} kWh/m2a (positive: the line guessed too low)")

nah = df[(df["kompaktheit"] - haus["kompaktheit"]).abs() < 0.25]
print()
print(f"{len(nah)} buildings within 0.25 of its compactness:")
print(f"  u-value from {nah['u_wert'].min():.2f} to {nah['u_wert'].max():.2f}, "
      f"average {nah['u_wert'].mean():.2f}")
print(f"  computed demand from {nah['hwb'].min():.1f} to {nah['hwb'].max():.1f}, "
      f"average {nah['hwb'].mean():.1f}")

### Your interpretation

1. **Size.** The typical miss is the square root of the error, back in kWh/m²a. The standard deviation is the typical miss of a model that always guesses the average demand. Both are printed. How much better than that is your line, and is it a good model, a poor one, or something in between?
2. **Sign and plausibility.** Your w is positive: a higher value of perimeter over square root of area goes with a higher demand. Say why that direction makes sense for a building. Then: is the size of w evidence from measured heating demand, or the result of fitting a line to computed values? Point to the sentence in this notebook that tells you.
3. **Units.** Give the units of w and of b. b is the line's value at compactness zero. Can a building have compactness zero? Say what that means for reading b on its own.
4. **Range.** Between which compactness values does your line rest on data? The second cell prints what it says at 15. Predicting outside the range of the data is called extrapolation. Do the buildings you fitted on support that number?
5. **Your building.** Give its residual with its sign, and say what the sign means. A higher u-value raises the computed demand, and the line knows nothing of it. Compare the u-value of your building with the average of its neighbours in compactness. Does that comparison explain the direction of the miss? If it does not, say so; that is a valid answer, as long as you say why it might not.
6. **Design consequence.** A colleague reads your w and concludes: lower a footprint's perimeter over square root of area by one, and you save that many kWh/m²a. Say what is right about that and what is wrong.

*Your text here.*